In [ ]:
!pip install -q kagglehub onnx onnxruntime

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import kagglehub
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

path = kagglehub.dataset_download("dextergoes/pothole-sensor-data")
rc_dir = os.path.join(path, "RoadCondition")
sensor_files = sorted(glob.glob(os.path.join(rc_dir, "*.csv")))

ACC_COLS = ["accelerometerX", "accelerometerY", "accelerometerZ"]
print(f"Dataset path: {path}")
print(f"Found {len(sensor_files)} RoadCondition files.")

Using Colab cache for faster access to the 'pothole-sensor-data' dataset.
Dataset path: /kaggle/input/pothole-sensor-data
Found 15 RoadCondition files.


In [ ]:
WINDOW_SIZE = 30
STRIDE = 5

X_list, y_list = [], []

for fpath in sensor_files:
    fname = os.path.basename(fpath).lower()
    if fname.startswith("bad"):
        label = 1.0
    elif fname.startswith("good"):
        label = 0.0
    else:
        continue

    df = pd.read_csv(fpath).dropna(subset=ACC_COLS)
    imu = df[ACC_COLS].values.astype(np.float32)

    for i in range(0, len(imu) - WINDOW_SIZE + 1, STRIDE):
        win = imu[i : i + WINDOW_SIZE]
        # Subtract window mean to remove gravity/DC offset (matches Simulink)
        win_norm = win - np.mean(win, axis=0, keepdims=True)
        X_list.append(win_norm)
        y_list.append(label)

X = np.array(X_list, dtype=np.float32)          # Shape: (N, 30, 3)
y = np.array(y_list, dtype=np.float32)[:, None] # Shape: (N, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train), torch.tensor(y_train)),
    batch_size=64, shuffle=True
)

print(f"Total windows: {len(X)} | Bad/Pothole: {int(y.sum())} | Good: {int(len(y) - y.sum())}")

Total windows: 2187 | Bad/Pothole: 393 | Good: 1794


In [ ]:
class PotholeLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=3, hidden_size=32, num_layers=1, batch_first=True)
        self.fc = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.sigmoid(self.fc(out[:, -1, :]))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotholeLSTM().to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([2.0]).to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

EPOCHS = 50
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        out, _ = model.lstm(bx)
        logits = model.fc(out[:, -1, :])
        loss = criterion(logits, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch:02d}/{EPOCHS} | Loss: {total_loss / len(train_loader):.4f}")

Epoch 10/50 | Loss: 0.7158
Epoch 20/50 | Loss: 0.3225
Epoch 30/50 | Loss: 0.2701
Epoch 40/50 | Loss: 0.2264
Epoch 50/50 | Loss: 0.2130


In [ ]:
model.eval()
with torch.no_grad():
    probs = model(torch.tensor(X_test).to(device)).cpu().numpy()
    preds = (probs > 0.5).astype(int)

print(classification_report(y_test.astype(int), preds, target_names=["Good Road", "Pothole/Bad"]))

              precision    recall  f1-score   support

   Good Road       0.97      0.97      0.97       359
 Pothole/Bad       0.86      0.86      0.86        79

    accuracy                           0.95       438
   macro avg       0.92      0.92      0.92       438
weighted avg       0.95      0.95      0.95       438



In [ ]:
import onnxruntime as ort

model.cpu().eval()
dummy_input = torch.randn(1, WINDOW_SIZE, 3)

torch.onnx.export(
    model,
    dummy_input,
    "pothole_lstm.onnx",
    input_names=["imu_window"],
    output_names=["pothole_prob"],
    opset_version=12,
    dynamo=False
)

# Verify ONNX inference matches
sess = ort.InferenceSession("pothole_lstm.onnx", providers=["CPUExecutionProvider"])
test_out = sess.run(None, {"imu_window": X_test[:1]})
print("Exported pothole_lstm.onnx successfully! Sample output:", test_out[0])

# Auto-download if running in Google Colab
try:
    from google.colab import files
    files.download("pothole_lstm.onnx")
except ImportError:
    print("Download pothole_lstm.onnx from your notebook's output directory.")

/tmp/ipykernel_2379/1758338848.py:6: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported pothole_lstm.onnx successfully! Sample output: [[0.0497494]]


/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:4496: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with LSTM can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>